# 使用 DistilBERT 进行影评情感分类

这个项目使用 IMDB 英文影评，判断一段评论是负面（`0`）还是正面（`1`）。我使用预训练的 DistilBERT，并添加自己的分类层，练习从文本处理到模型评估的完整 NLP 流程。

**流程：** 影评 → 分词与编码 → DistilBERT → `[CLS]` 向量 → 分类层 → 正负情感。

下面的解释对应实际代码。文末的结果来自原 notebook 的运行记录；这个整理版已清除执行输出，尚未重新训练。

## 1. 环境与数据

`datasets` 用来加载数据，`transformers` 提供预训练模型和 tokenizer，`torch` 用来搭建和训练分类器。首次运行需要联网下载依赖、数据和模型。

In [ ]:
%pip -q install torch datasets transformers

In [ ]:
import torch 
from datasets import load_dataset

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")

In [ ]:
dataset

## 2. 划分训练集与验证集

我先打乱原始训练集，再取 2,000 条用于训练、500 条用于验证，以减少实验所需的计算量。两部分互不重叠；`seed=42` 固定的是这次数据打乱的顺序。

训练集用于更新模型参数，验证集用于观察模型对未参与训练的数据的表现。原数据的测试集在本实验中尚未使用。

In [ ]:
#把数据做shuffle 处理
shuffled = dataset["train"].shuffle(seed = 42)
train_data = shuffled.select(range(2000))
val_data = shuffled.select(range(2000,2500))

## 3. 将文字转换成模型输入

Tokenizer 会把文本拆成 token（可能是单词或词的一部分），再转换成整数编号 `input_ids`。这些编号只是词表索引，本身不是语义向量。

本项目把输入长度统一为 128 个 token：过长的截断，过短的补齐。`attention_mask` 中的 `1` 表示有效位置，`0` 表示补齐位置，供模型在注意力计算时屏蔽补齐位置。

In [ ]:
#做tokenization
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased"
)


## 4. 从编号到上下文表示

我加载预训练的 DistilBERT，复用它学到的语言表示。Embedding 层将 token 编号映射为向量并结合位置信息；Transformer 再通过自注意力结合上下文，更新每个位置的表示。

下面用一条真实训练样本查看张量形状：`[1, 128, 768]` 分别表示 1 条文本、128 个 token 位置，以及每个位置的 768 维表示。前后形状相同，但表示的内容已经改变。

In [ ]:
import torch
from transformers import AutoModel

model_name = "distilbert/distilbert-base-uncased"
transformer = AutoModel.from_pretrained(model_name)

In [ ]:
encoded_review = tokenizer(
    train_data[0]["text"],
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

transformer.eval()
with torch.inference_mode():
    outputs = transformer(**encoded_review, output_hidden_states=True)

embeddings = outputs.hidden_states[0]
hidden_states = outputs.last_hidden_state

print("Input IDs:", encoded_review["input_ids"].shape)
print("Embeddings:", embeddings.shape)
print("Transformer Output:", hidden_states.shape)

## 5. 添加情感分类层

我取最后一层中第一个位置（`[CLS]`）的向量作为分类特征。这个向量会在情感分类训练中调整，并不是天然完美的句子摘要。

`Dropout(0.2)` 在训练时随机将部分特征置零，帮助减少过拟合。线性层把 768 维向量转换成两个类别分数（logits）；分数更高的类别就是预测结果。Logits 还不是概率。

训练时，优化器接收 `model.parameters()`，因此会同时更新 DistilBERT 和新增分类层，这属于**全模型微调**。

In [ ]:
import torch.nn as nn

class SentimentClassifier(nn.Module):
    def __init__(self,backbone):
        super().__init__()
        self.transformer = backbone
        self.dropout = nn.Dropout(p=0.2)
        self.fc = nn.Linear(768,2)

    def forward(self,input_ids,attention_mask):
        outputs = self.transformer(
            input_ids = input_ids,
            attention_mask = attention_mask
        )
        hidden_states = outputs.last_hidden_state
        cls_vector = hidden_states[:,0,:]
        x = self.dropout(cls_vector)
        logits = self.fc(x)
        return logits

In [ ]:
model = SentimentClassifier(transformer)
model.fc

## 6. 用损失衡量预测误差

`CrossEntropyLoss` 比较两个类别的 logits 与真实标签，衡量分类误差。它内部已处理所需的 log-softmax，不需要先手动做 softmax。

下面只演示损失的计算方式。分类层此时尚未训练，输出不能代表最终效果。

In [ ]:
criterion = nn.CrossEntropyLoss()
demo_inputs = tokenizer(
    ["I loved this movie.", "This movie was disappointing."],
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
demo_labels = torch.tensor([1, 0], dtype=torch.long)

model.eval()
with torch.inference_mode():
    demo_logits = model(**demo_inputs)
    demo_loss = criterion(demo_logits, demo_labels)

print("Example loss:", demo_loss.item())

## 7. 按批次准备数据

我把 token 编号、attention mask 和标签组合成 `TensorDataset`，再用 `DataLoader` 每次取出 8 条样本。

训练时打乱样本顺序，验证时保持顺序。一个训练批次中，输入形状为 `[8, 128]`，标签形状为 `[8]`。

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

MAX_LENGTH = 128
BATCH_SIZE = 8

def create_dataloader(data, shuffle=False):

    # Convert text into Token IDs
    texts = [str(x) if x is not None else "" for x in data["text"]]

    encodings = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    # Convert labels into Tensor
    labels = torch.tensor(
        list(data["label"]),
        dtype=torch.long
    )

    # Combine input_ids, attention_mask, labels
    tensor_dataset = TensorDataset(
        encodings["input_ids"],
        encodings["attention_mask"],
        labels
    )

    loader = DataLoader(
        tensor_dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle
    )

    return loader


train_loader = create_dataloader(train_data, shuffle=True)
val_loader = create_dataloader(val_data, shuffle=False)

input_ids, attention_mask, labels = next(iter(train_loader))

print("Input IDs:", input_ids.shape)
print("Attention Mask:", attention_mask.shape)
print("Labels:", labels.shape)

## 8. 训练与验证

我使用 AdamW，学习率为 `2e-5`，权重衰减为 `0.01`，训练 2 轮。每个批次依次执行：清空梯度 → 前向计算 → 计算损失 → 反向传播 → 更新参数。

`model.train()` 开启训练模式，包括 Dropout；`model.eval()` 切换到评估模式。验证阶段再用 `torch.inference_mode()` 关闭梯度记录，并通过 `argmax` 选择预测类别。

每轮结束后，只有验证损失降低时才保存权重到 `best_distilbert_imdb.pth`。这个文件需要运行训练后才会生成。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = 2e-5,
    weight_decay = 0.01
)


In [ ]:

EPOCHS = 2
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    # Training Mode
    model.train()

    train_loss = 0

    for input_ids, attention_mask, labels in train_loader:
        # 1. Clear previous gradients
        optimizer.zero_grad()

        # 2. Forward
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # 3. Compute Loss
        loss = criterion(logits, labels)

        # 4. Backpropagation
        loss.backward()

        # 5. Update Weights and Biases
        optimizer.step()

        train_loss += loss.item() * labels.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation
    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.inference_mode():

        for input_ids, attention_mask, labels in val_loader:

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, labels)

            val_loss += loss.item() * labels.size(0)

            predictions = logits.argmax(dim=1)

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    val_loss /= total
    val_accuracy = correct / total

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2%}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "best_distilbert_imdb.pth"
        )

        print("Best model saved!")


## 9. 结果与我的理解

以下数字来自**原 notebook 保存的运行记录**，不是这个整理版重新运行的结果。

| Epoch | 训练损失 | 验证损失 | 验证准确率 |
| --- | --- | --- | --- |
| 1 | 0.2934 | 0.3491 | 84.80% |
| 2 | 0.1276 | 0.4108 | 84.40% |

第二轮训练损失继续下降，但验证损失升高、准确率略降，出现了过拟合迹象。因此，按照验证损失选择模型时，应保留第一轮权重。

这个项目让我理解到：NLP 分类不只是把文字变成数字，还需要用上下文表示提取特征、通过损失训练模型，并用独立验证数据检查效果。训练损失更低，不一定代表泛化能力更好。

**局限：** 目前只使用小样本和一个验证划分；128 token 的截断可能丢失长影评后半段的信息；尚未在独立测试集评估。84.80% 只能作为这次验证实验的结果。后续可以加入 TF-IDF + Logistic Regression 基线，检查错误样本，并在选定模型后进行测试集评估。

## 参考资料

- [IMDB 数据集与标签定义](https://huggingface.co/datasets/stanfordnlp/imdb)
- [DistilBERT 官方文档](https://huggingface.co/docs/transformers/model_doc/distilbert)